# Rising Waters: Flood Prediction System

## SmartBridge Internship Project — Model Training Notebook

This notebook walks through the complete machine learning pipeline:
1. Data loading and exploration
2. Data visualization (univariate & multivariate)
3. Data preprocessing
4. Model building and comparison
5. Saving the best model, scaler, and feature names

## 1. Import Required Libraries

In [ ]:
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available. Using GradientBoostingClassifier as fallback.")

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline

## 2. Load Dataset

In [ ]:
# Set up paths relative to the project root
BASE_DIR = os.path.dirname(os.getcwd())
DATASET_PATH = os.path.join(BASE_DIR, "dataset", "flood dataset.xlsx")
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Load the Excel dataset
df = pd.read_excel(DATASET_PATH)
print(f"Dataset loaded from: {DATASET_PATH}")

## 3. Exploratory Data Analysis

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Dataset shape, info, and statistical summary
print("Shape:", df.shape)
df.info()
df.describe()

In [ ]:
# Missing value summary
missing = df.isnull().sum()
pd.DataFrame({"Missing Count": missing, "Missing %": (missing / len(df)) * 100})

### 3.1 Univariate Analysis — Histograms

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
n_cols = 3
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Distribution of {col}")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

### 3.2 Univariate Analysis — Box Plots (Outlier Detection)

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.boxplot(y=df[col], ax=axes[i], color="lightblue")
    axes[i].set_title(f"Box Plot: {col}")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

### 3.3 Multivariate Analysis — Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df[numerical_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Handle missing values: median for numerical, mode for categorical
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in [np.float64, np.int64]:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values after handling:", df.isnull().sum().sum())

In [ ]:
# Outlier handling using IQR capping (exclude target column)
for col in numerical_cols:
    if col == "flood":
        continue
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)

print("Outlier capping complete.")

In [ ]:
# Encode categorical columns if any exist
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
print(f"Categorical columns encoded: {cat_cols if cat_cols else 'None'}")

## 5. Feature-Target Split and Train-Test Split

In [ ]:
target_col = "flood"
feature_cols = [col for col in df.columns if col != target_col]
X = df[feature_cols]
y = df[target_col]

print(f"Features: {feature_cols}")
print(f"Target distribution:\n{y.value_counts()}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=10)
print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 6. Feature Scaling (StandardScaler)

In [ ]:
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)  # Fit on training data only
X_test_scaled = sc.transform(X_test)        # Transform test data (do NOT fit again)
print("Scaling complete.")

## 7. Model Training and Evaluation

In [ ]:
def xgboost_model():
    """Return XGBClassifier or GradientBoostingClassifier fallback."""
    if XGBOOST_AVAILABLE:
        return XGBClassifier(random_state=10, use_label_encoder=False, eval_metric="logloss")
    return GradientBoostingClassifier(random_state=10)

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=10),
    "Random Forest": RandomForestClassifier(random_state=10),
    "KNN": KNeighborsClassifier(),
    "XGBoost": xgboost_model(),
}

results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    trained_models[name] = model
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

## 8. Model Comparison

In [ ]:
comparison_df = pd.DataFrame({"Model": list(results.keys()), "Accuracy": list(results.values())})
comparison_df = comparison_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)
print("Model Accuracy Comparison:")
comparison_df

In [ ]:
# Visual comparison of model accuracies
plt.figure(figsize=(8, 5))
sns.barplot(data=comparison_df, x="Accuracy", y="Model", palette="Blues_d")
plt.title("Model Accuracy Comparison")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## 9. Save Best Model, Scaler, and Feature Names

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

joblib.dump(best_model, os.path.join(MODELS_DIR, "floods.save"))
joblib.dump(sc, os.path.join(MODELS_DIR, "transform.save"))
joblib.dump(feature_cols, os.path.join(MODELS_DIR, "features.save"))

print(f"Best model: {best_model_name} (Accuracy: {comparison_df.iloc[0]['Accuracy']:.4f})")
print("Saved: floods.save, transform.save, features.save")